# Diff-ICMH one-image Kaggle GPU smoke test

This notebook runs the repository's real `inference_partition.py` path from preflight through a downloadable output archive. Before choosing **Run All**, enable **Accelerator = GPU** and **Internet = On** in Kaggle notebook settings.

The default is deliberately small: `BPP_WEIGHT = 2`, the bundled `kodim01.png`, and 10 diffusion steps. This proves wiring while spending as little GPU time as practical. Set `SMOKE_TEST = False` after the smoke run to use the README-quality 50-step setting. The checkpoint downloads are multi-gigabyte, so the preflight stops before cloning or downloading when GPU, network, or at least 15 GiB of working space is unavailable.

## 1. Runtime preflight

This check uses only Kaggle's preinstalled Torch stack. It reports the detected ABI and verifies HTTPS access to both external model/code hosts before any expensive transfer.

In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys
from urllib.request import Request, urlopen

import torch
import torchvision

WORKING_DIR = Path('/kaggle/working')
if not WORKING_DIR.is_dir():
    raise RuntimeError(
        f'{WORKING_DIR} is unavailable. Run this notebook in Kaggle, or adapt WORKING_DIR explicitly.'
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. In Kaggle open Notebook options, select Accelerator = GPU, then Run All again.'
    )

free_gib = shutil.disk_usage(WORKING_DIR).free / 2**30
if free_gib < 15:
    raise RuntimeError(
        f'Only {free_gib:.1f} GiB is free under {WORKING_DIR}; at least 15 GiB is required before checkpoint downloads.'
    )

def check_https(url):
    request = Request(url, headers={'User-Agent': 'Wild-Diff-ICMH-Kaggle-preflight/1.0'})
    try:
        with urlopen(request, timeout=20) as response:
            status = getattr(response, 'status', 200)
            if status >= 400:
                raise RuntimeError(f'HTTP {status}')
    except Exception as exc:
        raise RuntimeError(
            f'HTTPS preflight failed for {url}: {exc}. Enable Internet in Kaggle notebook settings and retry.'
        ) from exc
    print(f'HTTPS OK: {url}')

for endpoint in ('https://github.com', 'https://huggingface.co'):
    check_https(endpoint)

BASE_TORCH_VERSION = torch.__version__
BASE_TORCHVISION_VERSION = torchvision.__version__
BASE_CUDA_VERSION = torch.version.cuda
GPU_NAME = torch.cuda.get_device_name(0)

print('Python:', sys.version.split()[0], platform.platform())
print('Torch:', BASE_TORCH_VERSION)
print('Torchvision:', BASE_TORCHVISION_VERSION)
print('Torch CUDA ABI:', BASE_CUDA_VERSION)
print('GPU:', GPU_NAME)
print(f'Free working space: {free_gib:.1f} GiB')
subprocess.run(['nvidia-smi'], check=False)

## 2. Acquire the inspected repository revision

The default `REPO_REF` is the exact revision inspected when this notebook was prepared. The detached checkout prevents a later branch update from silently changing the code used by the smoke test.

In [ ]:
REPO_URL = 'https://github.com/TranDuon/Wild-Diff-ICMH.git'
REPO_REF = 'aebaff6f61d0253e09e3f482d5887aa50e0a539a'
REPO_DIR = WORKING_DIR / 'Wild-Diff-ICMH'

def run_git(*args):
    return subprocess.run(['git', *args], cwd=REPO_DIR if REPO_DIR.exists() else WORKING_DIR, check=True, text=True)

if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository; remove or rename it before retrying.')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_DIR)],
        cwd=WORKING_DIR,
        check=True,
    )

run_git('remote', 'set-url', 'origin', REPO_URL)
run_git('fetch', '--depth', '1', 'origin', REPO_REF)
run_git('checkout', '--detach', '--force', REPO_REF)
checked_out_ref = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
if checked_out_ref != REPO_REF:
    raise RuntimeError(f'Expected {REPO_REF}, but checked out {checked_out_ref}.')

print('Repository:', REPO_DIR)
print('Pinned revision:', checked_out_ref)

## 3. Install inference dependencies without replacing Torch

Kaggle's preinstalled Torch, Torchvision, and CUDA determine the compatible binary ABI. The installs below do not request another Torch build. CompressAI and the vendored RAM++ package are installed with `--no-deps`; xFormers is selected from the PyTorch wheel index matching the detected CUDA ABI and is also installed with `--no-deps`. A failed xFormers wheel or import stops with the detected versions so the runtime can be changed deliberately instead of silently falling back.

In [ ]:
def pip_install(*packages, extra_args=()):
    command = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', *extra_args, *packages]
    print('Installing:', ' '.join(packages))
    subprocess.run(command, check=True)

# Runtime-safe replacements for stale repository pins.
pip_install(
    'numpy>=1.26,<2.0',
    'lightning>=2.6,<3',
    'pyiqa==0.1.15.post2',
    'fairscale==0.4.13',
    'einops>=0.8,<1',
    'kornia>=0.7,<1',
    'omegaconf==2.3.0',
    'open_clip_torch>=2.22,<3',
    'openai_clip>=1.0.1,<2',
    'opencv-python-headless>=4.8,<5',
    'Pillow>=10.4,<13',
    'pytorch-msssim>=1.0,<2',
    'scipy>=1.11,<2',
    'matplotlib>=3.8,<4',
    'tomli>=2,<3',
    'thop>=0.1.1.post2209072238',
    'timm>=0.9.7,<1.0',
    'tqdm>=4.66,<5',
    'transformers>=4.41,<5',
    'ultralytics_thop>=2,<3',
    'huggingface_hub>=0.23,<2',
)
pip_install('compressai==1.2.8', extra_args=('--no-deps',))
pip_install('-e', str(REPO_DIR / 'src' / 'recognize-anything'), extra_args=('--no-deps',))

if not BASE_CUDA_VERSION:
    raise RuntimeError(f'Torch {BASE_TORCH_VERSION} reports no CUDA ABI even though a GPU was detected.')
cuda_parts = BASE_CUDA_VERSION.split('.')
if len(cuda_parts) < 2 or not all(part.isdigit() for part in cuda_parts[:2]):
    raise RuntimeError(f'Cannot derive a PyTorch wheel index from CUDA ABI {BASE_CUDA_VERSION!r}.')
torch_wheel_tag = f'cu{cuda_parts[0]}{cuda_parts[1]}'
torch_wheel_index = f'https://download.pytorch.org/whl/{torch_wheel_tag}'
try:
    pip_install('xformers', extra_args=('--index-url', torch_wheel_index, '--no-deps'))
    subprocess.run(
        [sys.executable, '-c', 'import xformers, xformers.ops; print("xFormers:", xformers.__version__)'],
        check=True,
    )
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        'No working xFormers wheel was established for the detected Kaggle ABI. '
        f'Torch={BASE_TORCH_VERSION}, Torchvision={BASE_TORCHVISION_VERSION}, '
        f'CUDA={BASE_CUDA_VERSION}, index={torch_wheel_index}. '
        'Start a current Kaggle GPU image whose Torch/CUDA pair has a published PyTorch xFormers wheel, then retry.'
    ) from exc

## 4. Add notebook-local compatibility modules and smoke-test imports

The pinned repository imports legacy `pytorch_lightning` utility paths and the retired standalone `lpips` surface. The compatibility directory below maps only those call shapes to unified `lightning.pytorch` and `pyiqa`. It lives under `/kaggle/working`, is prepended to the inference subprocess `PYTHONPATH`, and does not modify repository sources. The subprocess import test runs before any model checkpoint download.

In [ ]:
import json
import textwrap

COMPAT_DIR = WORKING_DIR / 'difficmh_compat'
PL_DIR = COMPAT_DIR / 'pytorch_lightning'
UTILITIES_DIR = PL_DIR / 'utilities'
CALLBACKS_DIR = PL_DIR / 'callbacks'
for directory in (COMPAT_DIR, PL_DIR, UTILITIES_DIR, CALLBACKS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def write_compat(relative_path, content):
    target = COMPAT_DIR / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(textwrap.dedent(content).lstrip(), encoding='utf-8')
    return target

write_compat('pytorch_lightning/__init__.py', '''
    from lightning.pytorch import LightningDataModule, LightningModule, Trainer, seed_everything
    from lightning.pytorch import __version__

    __all__ = [
        'LightningDataModule', 'LightningModule', 'Trainer', 'seed_everything', '__version__'
    ]
''')
write_compat('pytorch_lightning/utilities/__init__.py', '''
    from .distributed import rank_zero_only
    from .types import EPOCH_OUTPUT, STEP_OUTPUT

    __all__ = ['rank_zero_only', 'EPOCH_OUTPUT', 'STEP_OUTPUT']
''')
write_compat('pytorch_lightning/utilities/distributed.py', '''
    from lightning.pytorch.utilities.rank_zero import rank_zero_only

    __all__ = ['rank_zero_only']
''')
write_compat('pytorch_lightning/utilities/types.py', '''
    from typing import Any, Dict, List, Mapping, Optional, Sequence, Union

    STEP_OUTPUT = Optional[Union[Dict[str, Any], Any]]
    EPOCH_OUTPUT = List[STEP_OUTPUT]

    __all__ = ['STEP_OUTPUT', 'EPOCH_OUTPUT']
''')
write_compat('pytorch_lightning/callbacks/__init__.py', '''
    from lightning.pytorch.callbacks import *
''')
write_compat('lpips.py', '''
    import torch
    import pyiqa

    class LPIPS(torch.nn.Module):
        def __init__(self, net='alex'):
            super().__init__()
            self.model = pyiqa.create_metric('lpips', net=net, device='cpu')

        def forward(self, img1, img2, normalize=False):
            # pyiqa's public metric API expects [0, 1]. Legacy lpips expects
            # [-1, 1] unless normalize=True, so convert only that legacy case.
            if not normalize:
                img1 = (img1 + 1.0) / 2.0
                img2 = (img2 + 1.0) / 2.0
            return self.model(img1, img2)
''')

compat_env = os.environ.copy()
pythonpath_parts = [str(COMPAT_DIR), str(REPO_DIR)]
if compat_env.get('PYTHONPATH'):
    pythonpath_parts.append(compat_env['PYTHONPATH'])
compat_env['PYTHONPATH'] = os.pathsep.join(pythonpath_parts)
compat_env['TOKENIZERS_PARALLELISM'] = 'false'

version_probe = textwrap.dedent(f'''
    import json, torch, torchvision
    import lightning, compressai, pyiqa, xformers
    assert torch.__version__ == {BASE_TORCH_VERSION!r}, (torch.__version__, {BASE_TORCH_VERSION!r})
    assert torchvision.__version__ == {BASE_TORCHVISION_VERSION!r}, (torchvision.__version__, {BASE_TORCHVISION_VERSION!r})
    print(json.dumps({{
        'torch': torch.__version__,
        'torchvision': torchvision.__version__,
        'cuda': torch.version.cuda,
        'lightning': lightning.__version__,
        'compressai': compressai.__version__,
        'pyiqa': pyiqa.__version__,
        'xformers': xformers.__version__,
    }}, indent=2))
''')
subprocess.run([sys.executable, '-c', version_probe], cwd=REPO_DIR, env=compat_env, check=True)
subprocess.run(
    [sys.executable, '-c', 'import inference_partition; print("inference_partition import smoke check: OK")'],
    cwd=REPO_DIR,
    env=compat_env,
    check=True,
)

## 5. Select the RD point and download exactly three checkpoints

`BPP_WEIGHT = 2` maps to the README checkpoint folder. The smoke path uses 10 steps; switching `SMOKE_TEST` to `False` selects the README-quality 50-step path. Downloads use `huggingface_hub` over normal verified HTTPS and request only the selected Diff-ICMH checkpoint.

In [ ]:
from huggingface_hub import hf_hub_download

BPP_WEIGHT = 2
CONTROL_MODULE_SCALE = 1.0
CFG_SCALE = 5.0
SMOKE_TEST = True
STEPS = 10 if SMOKE_TEST else 50
SEED = 231

if BPP_WEIGHT not in {2, 4, 8, 16, 32}:
    raise ValueError('BPP_WEIGHT must be one of 2, 4, 8, 16, or 32.')
FOLDER_NAME = f'CNscale1.0_1_1_{BPP_WEIGHT}_2_WTagGCM_bs16x1_lr0.00005_cfg7.0'

CHECKPOINTS_DIR = REPO_DIR / 'checkpoints'
SD_DIR = CHECKPOINTS_DIR / 'sd2p1'
RAM_DIR = CHECKPOINTS_DIR / 'ram'
SD_DIR.mkdir(parents=True, exist_ok=True)
RAM_DIR.mkdir(parents=True, exist_ok=True)

hf_hub_download(
    repo_id='Manojb/stable-diffusion-2-1-base',
    filename='v2-1_512-ema-pruned.ckpt',
    local_dir=SD_DIR,
)
hf_hub_download(
    repo_id='xinyu1205/recognize-anything-plus-model',
    filename='ram_plus_swin_large_14m.pth',
    local_dir=RAM_DIR,
)
hf_hub_download(
    repo_id='RuoyuFeng/Diff-ICMH',
    filename=f'difficmh_models/{FOLDER_NAME}/model.ckpt',
    local_dir=CHECKPOINTS_DIR,
)

CKPT_SD = SD_DIR / 'v2-1_512-ema-pruned.ckpt'
CKPT_RAM = RAM_DIR / 'ram_plus_swin_large_14m.pth'
CKPT_LC = CHECKPOINTS_DIR / 'difficmh_models' / FOLDER_NAME / 'model.ckpt'
minimum_sizes = {CKPT_SD: 1_000_000_000, CKPT_RAM: 100_000_000, CKPT_LC: 10_000_000}
for checkpoint, minimum_size in minimum_sizes.items():
    if not checkpoint.is_file() or checkpoint.stat().st_size < minimum_size:
        actual = checkpoint.stat().st_size if checkpoint.exists() else 0
        raise RuntimeError(
            f'Checkpoint is missing or unexpectedly small: {checkpoint} ({actual:,} bytes; expected at least {minimum_size:,}).'
        )
    print(f'Checkpoint OK: {checkpoint.relative_to(REPO_DIR)} ({checkpoint.stat().st_size / 2**30:.2f} GiB)')

## 6. Stage input images

The smoke run selects `data/kodak_subset/kodim01.png` explicitly. For a Kaggle dataset or a multi-image run, set `SMOKE_TEST = False`, point `SOURCE_DIR` at `/kaggle/input/...`, and choose a positive `MAX_IMAGES` or `None` for all discovered images. Only the notebook's dedicated staging/output directories are cleared.

In [ ]:
SOURCE_DIR = REPO_DIR / 'data' / 'kodak_subset'
# Example for an attached Kaggle dataset:
# SOURCE_DIR = Path('/kaggle/input/your-dataset/images')
MAX_IMAGES = 1 if SMOKE_TEST else None
IMAGE_PATTERNS = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.bmp')

if SMOKE_TEST:
    smoke_image = SOURCE_DIR / 'kodim01.png'
    if not smoke_image.is_file():
        raise FileNotFoundError(f'The pinned smoke image is missing: {smoke_image}')
    selected_images = [smoke_image]
else:
    discovered = sorted({path for pattern in IMAGE_PATTERNS for path in SOURCE_DIR.rglob(pattern)})
    if not discovered:
        raise FileNotFoundError(f'No supported images were found under {SOURCE_DIR}.')
    selected_images = discovered[:MAX_IMAGES] if MAX_IMAGES is not None else discovered

INPUT_DIR = WORKING_DIR / 'difficmh_input'
OUTPUT_DIR = WORKING_DIR / 'difficmh_output' / FOLDER_NAME
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
INPUT_DIR.mkdir(parents=True)
OUTPUT_DIR.mkdir(parents=True)

staged_sources = {}
for index, source in enumerate(selected_images):
    staged_name = f'{index:04d}_{source.name}'
    destination = INPUT_DIR / staged_name
    shutil.copy2(source, destination)
    staged_sources[staged_name] = source

print(f'Staged {len(staged_sources)} image(s) in {INPUT_DIR}')
for staged_name, source in staged_sources.items():
    print(f'  {staged_name} <- {source}')
print('Output directory:', OUTPUT_DIR)

## 7. Run the repository CLI

The command is an argument list, not a shell string. It passes both checkpoints, the actual model config, explicit input/output directories, seed/device/step settings, and all three README dotlist overrides.

In [ ]:
env = compat_env.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

command = [
    sys.executable, 'inference_partition.py',
    '--ckpt_sd', str(CKPT_SD),
    '--ckpt_lc', str(CKPT_LC),
    '--config', 'configs/model/diffeic.yaml',
    '--input', str(INPUT_DIR),
    '--output', str(OUTPUT_DIR),
    '--steps', str(STEPS),
    '--seed', str(SEED),
    '--device', 'cuda',
    f'params.control_stage_config.params.control_model_ratio={CONTROL_MODULE_SCALE}',
    'params.preprocess_tag_config.params.enabled=True',
    f'params.c_cfg_scale={CFG_SCALE}',
]

print('Running argument list:')
print(command)
subprocess.run(command, check=True, cwd=REPO_DIR, env=env)

## 8. Verify outputs, visualize, and recompute metrics with pyiqa

A successful CLI call must produce one reconstruction per staged image, at least one non-empty bitstream file, and `bpp.txt`. The CLI metrics are printed, then PSNR, SSIM, and LPIPS are recomputed through `pyiqa` for the displayed pair.

In [ ]:
import matplotlib.pyplot as plt
import textwrap
from PIL import Image

metrics_path = OUTPUT_DIR / 'bpp.txt'
if not metrics_path.is_file() or metrics_path.stat().st_size == 0:
    raise RuntimeError(f'Missing or empty CLI metric file: {metrics_path}')

result_paths = {name: OUTPUT_DIR / f'{Path(name).stem}.png' for name in staged_sources}
missing_results = [path for path in result_paths.values() if not path.is_file()]
if missing_results:
    raise RuntimeError(f'Missing reconstructed PNGs: {missing_results}')
stream_dir = OUTPUT_DIR / 'data'
stream_files = [path for path in stream_dir.rglob('*') if path.is_file() and path.stat().st_size > 0]
if not stream_files:
    raise RuntimeError(f'No non-empty bitstream files were created under {stream_dir}.')

print('CLI METRICS (bpp / PSNR / SSIM / LPIPS)')
print(metrics_path.read_text(encoding='utf-8'))
print(f'Non-empty bitstream files: {len(stream_files)}')

metric_program = textwrap.dedent('''
    import json
    import sys

    import pyiqa
    import torch
    from PIL import Image
    from torchvision.transforms.functional import pil_to_tensor

    original_path, reconstruction_path = sys.argv[1:3]
    device = torch.device('cuda')
    original = pil_to_tensor(Image.open(original_path).convert('RGB')).unsqueeze(0).float().to(device) / 255.0
    reconstruction = pil_to_tensor(Image.open(reconstruction_path).convert('RGB')).unsqueeze(0).float().to(device) / 255.0

    metrics = {
        'PSNR': float(pyiqa.create_metric('psnr', device=device)(original, reconstruction).item()),
        'SSIM': float(pyiqa.create_metric('ssim', device=device)(original, reconstruction).item()),
        'LPIPS': float(pyiqa.create_metric('lpips', net='alex', device=device)(original, reconstruction).item()),
    }
    print(json.dumps(metrics, sort_keys=True))
''')

for staged_name, result_path in result_paths.items():
    original_path = INPUT_DIR / staged_name
    original = Image.open(original_path).convert('RGB')
    reconstruction = Image.open(result_path).convert('RGB')

    figure, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(original)
    axes[0].set_title(f'Original: {staged_name}')
    axes[1].imshow(reconstruction)
    axes[1].set_title(f'Diff-ICMH reconstruction ({STEPS} steps)')
    for axis in axes:
        axis.axis('off')
    plt.tight_layout()
    plt.show()

    metric_process = subprocess.run(
        [sys.executable, '-c', metric_program, str(original_path), str(result_path)],
        cwd=REPO_DIR,
        env=compat_env,
        check=True,
        text=True,
        capture_output=True,
    )
    print(f'pyiqa metrics for {staged_name}: {metric_process.stdout.strip()}')


## 9. Export a downloadable ZIP

The archive root is the dedicated `OUTPUT_DIR`, so it contains only reconstructions, bitstreams, and metrics from this run.

In [ ]:
from IPython.display import FileLink, display

archive_base = WORKING_DIR / f'difficmh_bpp{BPP_WEIGHT}_{STEPS}steps'
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR))
if not archive_path.is_file() or archive_path.stat().st_size == 0:
    raise RuntimeError(f'ZIP export failed: {archive_path}')
print(f'Archive ready: {archive_path} ({archive_path.stat().st_size / 2**20:.1f} MiB)')
display(FileLink(str(archive_path)))

## Validation status

The notebook file is locally validated only for JSON structure, Python syntax, empty saved outputs, and static checkpoint/config/CLI wiring. That does **not** prove Kaggle execution. End-to-end Kaggle status remains unverified until **Run All** reaches the ZIP cell with a reconstruction, non-empty bitstreams, and printed bpp/PSNR/SSIM/LPIPS values.